# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulmoiz-25/FlyRank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing **Refresh / Content Opportunity Scoring** as my provisional lane.

The goal of this project is to identify which content pages should be reviewed first using observable SEO signals such as **impressions, content age, CTR, average position, and trend direction**. Instead of reviewing pages randomly, content editors can prioritize the pages that appear to have the greatest opportunity for improvement.

I selected this lane because the starter dataset already contains several useful observable signals that can help create a practical review queue. This project is intended to provide **decision support** rather than replace human judgment, and I may refine the exact approach after exploring the larger warehouse dataset.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane = "Refresh / Content Opportunity Scoring"
unit_of_analysis = "One anonymized content page"
output = "Ranked review queue"

print("Lane:", lane)
print("Unit of analysis:", unit_of_analysis)
print("Output:", output)


Lane: Refresh / Content Opportunity Scoring
Unit of analysis: One anonymized content page
Output: Ranked review queue


## 2. The question: decision, action, cost of a wrong call

### Research Question

**Which content pages should be reviewed first based on observable SEO signals such as impressions, CTR, average position, content age, and trend direction?**

The **unit of analysis** is one anonymized content page. The expected **output** is a ranked review queue that helps editors decide whether a page should be **refreshed, expanded, monitored, or left unchanged**.

The decision this work supports is how editors prioritize limited review time. A **false positive** wastes editor time by recommending unnecessary work, while a **false negative** may miss a valuable declining page that should have been reviewed. Therefore, the quality of the highest-ranked recommendations is especially important, making **Precision@K** an appropriate evaluation metric.

This project aims to support decision-making using measurable data rather than replacing human expertise.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
decision = {
    "Decision": "Which pages should be reviewed first?",
    "Actor": "Content Editor",
    "Action": "Refresh, Expand, Monitor or Leave Unchanged",
    "False Positive": "Wasted editor time",
    "False Negative": "Missed declining page",
    "Evaluation Metric": "Precision@K"
}

for k, v in decision.items():
    print(f"{k}: {v}")


Decision: Which pages should be reviewed first?
Actor: Content Editor
Action: Refresh, Expand, Monitor or Leave Unchanged
False Positive: Wasted editor time
False Negative: Missed declining page
Evaluation Metric: Precision@K


## 3. Quick look at the data

The starter dataset contains **30,000 anonymized content pages** collected from **32 clients**, providing enough variety to explore content refresh opportunities.

The exploratory analysis shows that **13,152 pages are both declining and still receiving at least 100 impressions**, meaning they remain visible but may benefit from review. It also identifies **9,759 pages** with at least **500 impressions**, positions between **1 and 20**, and **CTR below 0.5%**, suggesting additional opportunities for optimization.

These observations indicate that relying on a single metric is unlikely to be sufficient. Combining multiple observable signals such as impressions, CTR, average position, content age, and trend direction may produce a more useful review queue for content editors.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the starter CSV robustly in Colab, whether the notebook was opened
# from a cloned repo or directly from GitHub.

from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found.")

df = pd.read_csv(data_path)

required = {
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
    "avg_position",
    "ctr"
}

missing = required.difference(df.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

total_pages = df["content_id"].nunique()
total_clients = df["client_id"].nunique()

eligible = df[
    (df["impressions_90d"].fillna(0) > 0) &
    (df["content_age_days"].fillna(0) >= 90)
].drop_duplicates("content_id")

declining_with_demand = eligible[
    (eligible["trend_direction"].str.lower() == "down") &
    (eligible["impressions_90d"] >= 100)
]

visible_low_ctr = eligible[
    (eligible["impressions_90d"] >= 500) &
    (eligible["avg_position"] > 0) &
    (eligible["avg_position"] <= 20) &
    (eligible["ctr"] < 0.5)
]

print(f"Rows: {len(df):,}")
print(f"Unique Pages: {total_pages:,}")
print(f"Clients: {total_clients:,}")
print(f"Eligible Pages: {len(eligible):,}")
print(f"Declining Pages with Demand: {len(declining_with_demand):,}")
print(f"Visible Low CTR Pages: {len(visible_low_ctr):,}")

Rows: 30,000
Unique Pages: 30,000
Clients: 32
Eligible Pages: 30,000
Declining Pages with Demand: 13,152
Visible Low CTR Pages: 9,759


## 4. Careful words: what I can and can't claim

This project identifies **observed relationships** between measurable SEO signals and declining content. The results are intended to **support editor decisions** by ranking pages that may deserve review based on observable search and engagement data.

This project **cannot prove causation**, identify Google's ranking factors, or guarantee future traffic growth. Any recommendations produced by the model should be treated as **decision-support** rather than proof that refreshing a page will improve its performance.

Future work should validate the approach using proper train/test splits or client-holdout evaluation so that recommendations generalize to unseen data.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
allowed = [
    "Observed relationship",
    "Directional pattern",
    "Decision-support",
    "Measured on validation data"
]

not_allowed = [
    "Google ranking factor",
    "Guaranteed traffic growth",
    "Proof of causation"
]

print("Allowed language:")
for item in allowed:
    print("-", item)

print("\nClaims to avoid:")
for item in not_allowed:
    print("-", item)

Allowed language:
- Observed relationship
- Directional pattern
- Decision-support
- Measured on validation data

Claims to avoid:
- Google ranking factor
- Guaranteed traffic growth
- Proof of causation


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.